In [0]:
# Session restarted -> current schema reset to default.
# Re-point the session at our schema before using short table names.
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA lakehouse")

print(spark.catalog.currentCatalog(), spark.catalog.currentDatabase())

workspace lakehouse


In [0]:
# read the table into a DataFrame — this is a LAZY operation, no data is read yet
df = spark.read.table("samples.nyctaxi.trips")

# schema is cheap: Spark reads only metadata, not the rows
df.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



In [0]:
# .show() is an ACTION — this is what triggers the computation
df.show(5)

# execution plan: what Spark PLANS to do when asked
df.explain()

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|
+--------------------+---------------------+-------------+-----------+----------+-----------+
only showing top 5 rows
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet samples.nyctaxi.trips[tpep_pickup_datetime#1

In [0]:
from pyspark.sql import functions as F

result = (
    df.groupBy("pickup_zip")                                 # group by pickup zip code
      .agg(
          F.round(F.avg("fare_amount"), 2).alias("avg_fare"), # average fare
          F.count("*").alias("trips")                         # number of trips
      )
      .orderBy(F.desc("trips"))                               # busiest areas first
)

result.show(10)

+----------+--------+-----+
|pickup_zip|avg_fare|trips|
+----------+--------+-----+
|     10001|   10.62| 1227|
|     10003|   10.98| 1181|
|     10011|   10.91| 1129|
|     10021|   10.21| 1021|
|     10018|    11.4| 1012|
|     10023|   10.04| 1008|
|     10028|   10.21|  929|
|     10012|   11.35|  834|
|     10110|    10.9|  763|
|     10065|    9.81|  702|
+----------+--------+-----+
only showing top 10 rows


In [0]:
# Phase 2: medallion architecture (bronze -> silver -> gold)
# Create our own schema to hold the pipeline tables.
# Free Edition provides a writable "workspace" catalog by default.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.lakehouse")
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA lakehouse")

# confirm where we are now
print(spark.catalog.currentCatalog(), spark.catalog.currentDatabase())

workspace lakehouse


In [0]:
# Bronze = raw layer: ingest the source as-is, no cleaning yet.
# In a real pipeline this is where the raw stream lands untouched;
# here we simulate ingestion by copying the sample table.
bronze = spark.read.table("samples.nyctaxi.trips")

# Write it as our own managed Delta table.
(bronze.write
    .format("delta")       # Delta is the default on Databricks; shown here explicitly
    .mode("overwrite")     # replace if it already exists — makes the cell safe to re-run
    .saveAsTable("bronze_trips"))

print("bronze_trips rows:", spark.table("bronze_trips").count())

bronze_trips rows: 21932


In [0]:
# Delta keeps a transaction log: every write is an atomic, versioned commit.
# DESCRIBE HISTORY shows each version — this is what powers "time travel".
display(spark.sql("DESCRIBE HISTORY bronze_trips"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-01T13:51:01.000Z,76580100620809,agvanuk.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3467887741235014),8457e18b-464c-49ae-ae07-389c47dcfaaa,0801-134909-ssah643a-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 21932, numOutputBytes -> 315164)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
from pyspark.sql import functions as F

# Silver = cleaned layer: apply data-quality rules to the raw bronze data.
silver = (
    spark.table("bronze_trips")
        # keep only rows that make physical sense
        .filter(F.col("trip_distance") > 0)          # drop zero-distance trips
        .filter(F.col("fare_amount") > 0)            # drop non-positive fares
        .filter(F.col("fare_amount") < 500)          # drop absurd outliers
        # derive trip duration in minutes from the two timestamps
        .withColumn(
            "duration_min",
            (F.col("tpep_dropoff_datetime").cast("long")
             - F.col("tpep_pickup_datetime").cast("long")) / 60
        )
        .filter(F.col("duration_min") > 0)           # drop trips that end before they start
)

# Persist as a managed Delta table (our silver layer).
(silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_trips"))

# compare row counts: how many rows did cleaning remove?
bronze_count = spark.table("bronze_trips").count()
silver_count = spark.table("silver_trips").count()
print(f"bronze: {bronze_count}  ->  silver: {silver_count}  (removed {bronze_count - silver_count})")

bronze: 21932  ->  silver: 21847  (removed 85)


In [0]:
# Look at the history BEFORE we change anything.
print("=== history before ===")
display(spark.sql("DESCRIBE HISTORY silver_trips"))

=== history before ===


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-01T13:54:34.000Z,76580100620809,agvanuk.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3467887741235014),8d8e1183-41b2-4752-92d4-1878cddb9899,0801-134909-ssah643a-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 354417)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
from pyspark.sql import functions as F

# Deliberately overwrite silver with a NARROWER filter (stricter cleaning).
# This creates version 1 in the Delta log.
stricter = (
    spark.table("bronze_trips")
        .filter(F.col("trip_distance") > 0)
        .filter(F.col("fare_amount") > 0)
        .filter(F.col("fare_amount") < 500)
        .filter(F.col("trip_distance") < 50)   # NEW rule: drop very long trips too
        .withColumn(
            "duration_min",
            (F.col("tpep_dropoff_datetime").cast("long")
             - F.col("tpep_pickup_datetime").cast("long")) / 60
        )
        .filter(F.col("duration_min") > 0)
)

(stricter.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_trips"))

print("current row count:", spark.table("silver_trips").count())

current row count: 21847


In [0]:
# Now the magic: read the table AS IT WAS at version 0, in one line.
v0 = spark.read.option("versionAsOf", 0).table("silver_trips")
v1 = spark.table("silver_trips")   # current version

print("version 0 rows:", v0.count())
print("version 1 rows (current):", v1.count())

version 0 rows: 21847
version 1 rows (current): 21847


In [0]:
# Confirm the Delta log holds two versions now (0 and 1).
display(spark.sql("DESCRIBE HISTORY silver_trips"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-01T13:56:50.000Z,76580100620809,agvanuk.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3467887741235014),0a90a575-979f-4a25-b145-912c5d6e2e39,0801-134909-ssah643a-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 354417, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 354417)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-01T13:54:34.000Z,76580100620809,agvanuk.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3467887741235014),8d8e1183-41b2-4752-92d4-1878cddb9899,0801-134909-ssah643a-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 354417)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
from pyspark.sql import functions as F

# Simulate a mistake: overwrite silver with a bad filter that keeps only
# high-fare trips. This clearly changes the data and creates version 2.
bad = spark.table("silver_trips").filter(F.col("fare_amount") > 20)
bad.write.format("delta").mode("overwrite").saveAsTable("silver_trips")

# Compare the ORIGINAL version 0 against the current (broken) table.
# SQL "VERSION AS OF" is the clearest time-travel syntax.
display(spark.sql("""
    SELECT 'v0 (original)'    AS state, count(*) AS rows FROM silver_trips VERSION AS OF 0
    UNION ALL
    SELECT 'current (broken)' AS state, count(*) AS rows FROM silver_trips
"""))

state,rows
v0 (original),21847
current (broken),2781


In [0]:
# Recover: roll the table back to the good version in a single command.
spark.sql("RESTORE TABLE silver_trips TO VERSION AS OF 1")

print("silver_trips rows after restore:", spark.table("silver_trips").count())

silver_trips rows after restore: 21847


In [0]:
from pyspark.sql import functions as F

# Gold = business-ready layer: aggregates built for a specific use case
# (here: a per-zone summary for a dashboard).
# Built on SILVER, so it uses only clean rows and can reuse the derived duration_min.
gold = (
    spark.table("silver_trips")
        .groupBy("pickup_zip")
        .agg(
            F.count("*").alias("trips"),                           # number of trips
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),    # average fare
            F.round(F.avg("trip_distance"), 2).alias("avg_miles"), # average distance
            F.round(F.avg("duration_min"), 1).alias("avg_minutes") # avg duration (from silver)
        )
        .orderBy(F.desc("trips"))
)

# Persist as a managed Delta table (our gold layer).
(gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_zone_summary"))

display(spark.table("gold_zone_summary"))

pickup_zip,trips,avg_fare,avg_miles,avg_minutes
10001,1227,10.62,2.21,14.5
10003,1180,10.99,2.33,12.4
10011,1128,10.92,2.29,15.0
10021,1017,10.15,2.03,11.5
10018,1010,11.42,2.58,15.6
10023,1006,10.05,2.12,12.4
10028,927,10.18,2.21,12.3
10012,831,11.38,2.44,13.2
10110,761,10.86,2.31,12.2
10065,700,9.77,1.97,16.2


In [0]:
# All three medallion layers now exist as Delta tables in our schema.
display(spark.sql("SHOW TABLES IN workspace.lakehouse"))

database,tableName,isTemporary
lakehouse,bronze_trips,false
lakehouse,gold_zone_summary,false
lakehouse,silver_trips,false
